Developing a machine learning model using Python to predict the number of calories burnt during a workout based on some biological measures.

In [1]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#for modeling and validation
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#algorithms
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

#plotting style
sns.set_theme(style="whitegrid")

In [8]:
#load dataset
cal_df=pd.read_csv('calories.csv')
print("Calorie df first 5 rows:")
print(cal_df.head())

ex_df=pd.read_csv('exercise.csv')
print("Exercise df first 5 rows:")
print(ex_df.head())

Calorie df first 5 rows:
    User_ID  Calories
0  14733363     231.0
1  14861698      66.0
2  11179863      26.0
3  16180408      71.0
4  17771927      35.0
Exercise df first 5 rows:
    User_ID  Gender  Age  Height  Weight  Duration  Heart_Rate  Body_Temp
0  14733363    male   68   190.0    94.0      29.0       105.0       40.8
1  14861698  female   20   166.0    60.0      14.0        94.0       40.3
2  11179863    male   69   179.0    79.0       5.0        88.0       38.7
3  16180408  female   34   179.0    71.0      13.0       100.0       40.5
4  17771927  female   27   154.0    58.0      10.0        81.0       39.8


In [10]:
#merge datasets
cal_ex_df=pd.merge(cal_df,ex_df,on='User_ID')
cal_ex_df.head(10)

,User_ID,Calories,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,14733363,231.0,male,68,190.0,94.0,29.0,105.0,40.8
1,14861698,66.0,female,20,166.0,60.0,14.0,94.0,40.3
2,11179863,26.0,male,69,179.0,79.0,5.0,88.0,38.7
3,16180408,71.0,female,34,179.0,71.0,13.0,100.0,40.5
4,17771927,35.0,female,27,154.0,58.0,10.0,81.0,39.8
5,15130815,123.0,female,36,151.0,50.0,23.0,96.0,40.7
6,19602372,112.0,female,33,158.0,56.0,22.0,95.0,40.5
7,11117088,143.0,male,41,175.0,85.0,25.0,100.0,40.7
8,12132339,134.0,male,60,186.0,94.0,21.0,97.0,40.4
9,17964668,72.0,female,26,146.0,51.0,16.0,90.0,40.2


In [12]:
#drop User_ID 
cal_ex_df.drop(columns=['User_ID'], inplace=True)

In [14]:
#quality checks
print(f"Dataset Shape: {cal_ex_df.shape}")
print(f"\nMissing Values:\n{cal_ex_df.isnull().sum()}")
print(f"\nDuplicates: {cal_ex_df.duplicated().sum()}")

Dataset Shape: (15000, 8)

Missing Values:
Calories      0
Gender        0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
dtype: int64

Duplicates: 1


In [15]:
#clean duplicates
initial_count = len(cal_ex_df)
cal_ex_df.drop_duplicates(inplace=True)
print(f"Removed {initial_count - len(cal_ex_df)} duplicate row(s). Cleaned shape: {cal_ex_df.shape}")

Removed 1 duplicate row(s). Cleaned shape: (14999, 8)


In [ ]:
#feature engineering
def engineer_features(data:pd.DataFrame)->pd.DataFrame:
    cal_ex_df_feat=data.copy()
    cal_ex_df_feat['BMI']=cal_ex_df_feat['W']